In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [3]:
customer_name = 'Cadent'
customer_id = Query(query = f"SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'").execute([KPIHub_Conn]).values[0][0]


tableList = [KPI_ReportSummary]
aggregator = ['ReportYear', 'ReportWeek']
data = {}
for table in tableList:
    data[table] = Query(query = f"SELECT * FROM {table.name} WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'))").execute([KPIHub_Conn])

def report_summary_apply(df):
    return pd.Series({
        'FOVMain': df['DistributionPipeCoveredKm'].sum()/df['DistributionPipeKm'].sum(),
        'ReportAssetLengthKm': df['ReportAssetLengthKm'].sum() if 'ReportAssetLengthKm' in df else None,
        'AssetCoveredLengthKm': df['AssetCoveredLengthKm'].sum() if 'AssetCoveredLengthKm' in df else None,
        'DistributionPipeKm': df['DistributionPipeKm'].sum() if 'DistributionPipeKm' in df else None,
        'DistributionPipeCoveredKm': df['DistributionPipeCoveredKm'].sum() if 'DistributionPipeCoveredKm' in df else None,
        'ServicePipeKm': df['ServicePipeKm'].sum() if 'ServicePipeKm' in df else None,
        'ServicePipeCoveredKm': df['ServicePipeCoveredKm'].sum() if 'ServicePipeCoveredKm' in df else None,
        'ReportCount': df.shape[0]
    })

report_by_period = data[KPI_ReportSummary].groupby(aggregator).apply(report_summary_apply)
report_by_period = report_by_period.round(2)

/tmp/ipykernel_1019217/1637010912.py:13: RuntimeWarning: invalid value encountered in double_scalars
  'FOVMain': df['DistributionPipeCoveredKm'].sum()/df['DistributionPipeKm'].sum(),


In [4]:
r = report_by_period.reset_index()
r_long = r.melt(id_vars=aggregator, var_name='KPIId', value_name='Value')
r_long['Id'] = r_long.apply(lambda row: f"{row['KPIId']}_{customer_name}_Y{row['ReportYear']}_W{row['ReportWeek']}", axis=1)
r_long = r_long.rename(columns={'ReportWeek': 'PeriodValue', 'ReportYear': 'Year'})
r_long['PeriodType'] = "Weekly"
r_long['LastUpdated'] = datetime.now()
r_long['CustomerId'] = customer_id


In [5]:
KPI_Data.update_table(arguments = {'DataFrame': r_long, 'db_path': DB_PATH, 'PrimaryKey': 'Id'})

In [6]:
KPI_Data.query_table(arguments = {'db_path': DB_PATH})

,Id,KPIId,CustomerId,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_Y2023_W14,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2023,Weekly,14,None,None,2026-06-22 09:57:37.990053
1,FOVMain_Cadent_Y2023_W15,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2023,Weekly,15,None,None,2026-06-22 09:57:37.990053
2,FOVMain_Cadent_Y2023_W16,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2023,Weekly,16,None,None,2026-06-22 09:57:37.990053
3,FOVMain_Cadent_Y2023_W17,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2023,Weekly,17,None,None,2026-06-22 09:57:37.990053
4,FOVMain_Cadent_Y2023_W19,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2023,Weekly,19,None,None,2026-06-22 09:57:37.990053
...,...,...,...,...,...,...,...,...,...
7971,DistributionPipeKm_Cadent_Y2026_W26,DistributionPipeKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,26,390.62,None,2026-06-22 09:57:37.990053
7972,DistributionPipeCoveredKm_Cadent_Y2026_W26,DistributionPipeCoveredKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,26,359.19,None,2026-06-22 09:57:37.990053
7973,ServicePipeKm_Cadent_Y2026_W26,ServicePipeKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,26,14.91,None,2026-06-22 09:57:37.990053
7974,ServicePipeCoveredKm_Cadent_Y2026_W26,ServicePipeCoveredKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,26,11.57,None,2026-06-22 09:57:37.990053
